In [4]:
!pwd

/workspace/160305


In [1]:
import time
import threading
from pathlib import Path

import cv2
import numpy as np
import ipywidgets.widgets as widgets
from IPython.display import display

import onnxruntime as ort
from jetbot import Camera
from PUTDriver import PUTDriver

In [2]:
MODEL_PATH = './jetbot_model.onnx'

# no external configs, just this
config = {
    'model': { 'path': MODEL_PATH },
    'robot': {
        'max_speed':    0.93,    # current best 0.93
        'max_steering': 0.5,
        'differential': { 'left': 1.0, 'right': 0.995 },   # best calibration
    },
}
driver = PUTDriver(config=config)

camera = Camera.instance()


In [3]:
sess = ort.InferenceSession(config['model']['path'], providers=['CPUExecutionProvider'])
input_name  = sess.get_inputs()[0].name
input_shape = sess.get_inputs()[0].shape
print('onnx loaded:', config['model']['path'])
print('input :', input_name, input_shape)
print('output:', sess.get_outputs()[0].name, sess.get_outputs()[0].shape)

OLD_MAX      = 0.20
NEW_MAX      = 0.40
NORMAL_SCALE = OLD_MAX / NEW_MAX

LATENCY_K = 2   # MUST match training CFG['latency_k']


def preprocess(image_bgr):
    """camera BGR HxWx3 uint8 -> (1, 3, 224, 224) float32 RGB in [0,1]."""
    img = cv2.resize(image_bgr, (224, 224), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    return np.expand_dims(img, 0)


def postprocess(output):
    """onnx output (1, 2) -> (forward, left) clipped to [-1, 1]."""
    out = np.clip(output[0], -1.0, 1.0)
    return float(out[0]), float(out[1])


# warm-up so the first real frame isn't slower than the rest
_ = sess.run(None, {input_name: np.zeros((1, 3, 224, 224), dtype=np.float32)})
print(f'warm-up done | OLD_MAX={OLD_MAX} NEW_MAX={NEW_MAX} NORMAL_SCALE={NORMAL_SCALE:.2f} LATENCY_K={LATENCY_K}')


onnx loaded: ./jetbot_model.onnx
input : input [1, 3, 224, 224]
output: output [1, 2]
warm-up done | OLD_MAX=0.2 NEW_MAX=0.4 NORMAL_SCALE=0.50 LATENCY_K=2


In [4]:
# UI:
#   autonomy toggle  - master ON/OFF (off bu default)
#   ctrl_delay  - delay the applied control by N frames. 0 = full look-ahead as trained (LATENCY_K frames ahead)
autonomy_button = widgets.ToggleButton(
    description='Start/Stop Autonomy', button_style='success', value=False,
)
forward_cap_slider = widgets.FloatSlider(
    value=NORMAL_SCALE, min=0.0, max=1.0, step=0.05,
    description='forward cap:', continuous_update=True,
)
ctrl_delay_slider = widgets.IntSlider(
    value=0, min=0, max=LATENCY_K, step=1,
    description='ctrl delay:', continuous_update=True,
)
fps_widget   = widgets.FloatText(description='inference Hz:', value=0.0)
debug_widget = widgets.HTML(value="<b>autonomy OFF</b>")

display(widgets.VBox([
    widgets.HBox([autonomy_button, forward_cap_slider, ctrl_delay_slider, fps_widget]),
    debug_widget,
]))


In [5]:
from collections import deque

run_thread = True
loop_period = 0.1   # target 10 Hz to match the data-collection rate


def autonomy_loop():
    last_print = time.time()
    ema_hz = 0.0
    control_buffer = deque(maxlen=LATENCY_K + 1)

    while run_thread:
        t0 = time.time()
        frame = camera.value

        if frame is None:
            driver.update(0.0, 0.0)
            time.sleep(loop_period)
            continue

        x = preprocess(frame)
        out = sess.run(None, {input_name: x})[0]
        forward, left = postprocess(out)

        # buffer the fresh prediction, then pick the one =delay frames back
        # delay=0 means newest (full look-ahead)
        control_buffer.append((forward, left))
        delay = int(ctrl_delay_slider.value)
        forward, left = control_buffer[-1 - min(delay, len(control_buffer) - 1)]

        if autonomy_button.value:
            cap = float(forward_cap_slider.value)
            capped_forward = max(-cap, min(forward, cap))   # symmetric clip
            driver.update(capped_forward, left)
            tag = 'TURBO' if capped_forward > NORMAL_SCALE + 1e-3 else 'CRUISE'
            status = (
                f"<b style='color:green;'>AUTO/{tag}</b> "
                f"fwd={capped_forward:+.2f} (raw {forward:+.2f}, cap {cap:.2f}) "
                f"left={left:+.2f} delay={delay}"
            )
        else:
            driver.update(0.0, 0.0)
            status = f"<b>autonomy OFF</b> (would: fwd={forward:+.2f} left={left:+.2f} delay={delay})"

        dt = time.time() - t0
        inst_hz = 1.0 / max(dt, 1e-6)
        ema_hz = 0.9 * ema_hz + 0.1 * inst_hz if ema_hz > 0 else inst_hz
        if time.time() - last_print > 0.5:
            fps_widget.value = round(ema_hz, 1)
            last_print = time.time()

        debug_widget.value = status

        slack = loop_period - (time.time() - t0)
        if slack > 0:
            time.sleep(slack)

    driver.update(0.0, 0.0)


thread = threading.Thread(target=autonomy_loop)
thread.start()
print('autonomy thread started — toggle the button to enable driving')


autonomy thread started — toggle the button to enable driving


In [6]:
# clean shutdown
run_thread = False
thread.join()
driver.update(0.0, 0.0)
camera.stop()
debug_widget.value = "<b>autonomy stopped, motors at 0.</b>"